# VayuVision — Feature Engineering

## Objective

This notebook creates calendar and time-series features for next-day AQI forecasting.

The model will use information available today and on previous days to predict the next day's AQI.

## Features created

### Calendar features
- year
- month
- day_of_week
- is_weekend

### Time-series features
- aqi_lag_1
- aqi_lag_7
- aqi_rolling_mean_3
- aqi_rolling_mean_7
- pm25_lag_1
- pm10_lag_1

### Prediction target
- next_day_aqi

All time-series features are calculated separately for each city to prevent information from one city leaking into another.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_PATH = Path("data/processed/cleaned_city_day.csv")

if not PROCESSED_PATH.exists():
    PROCESSED_PATH = Path("../data/processed/cleaned_city_day.csv")

feature_df = pd.read_csv(PROCESSED_PATH)

feature_df["Date"] = pd.to_datetime(
    feature_df["Date"],
    errors="coerce"
)

feature_df = (
    feature_df
    .sort_values(["City", "Date"])
    .reset_index(drop=True)
)

print("Cleaned dataset shape:", feature_df.shape)
print("Duplicate City-Date pairs:", feature_df.duplicated(["City", "Date"]).sum())

display(feature_df.head())

Cleaned dataset shape: (4684, 20)
Duplicate City-Date pairs: 0


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,missing_value_count,is_aqi_outlier,is_pm25_outlier,data_quality_score
0,Bengaluru,2015-03-21,48.59,77.36,3.47,27.00,18.04,28.02,3.89,1.94,52.96,21.33,196.72,NaN,91.0,Satisfactory,2,False,False,90
1,Bengaluru,2015-03-22,47.38,77.36,2.84,22.39,15.33,23.19,11.29,2.05,71.85,14.97,138.20,NaN,120.0,Moderate,2,False,False,90
2,Bengaluru,2015-03-23,65.65,77.36,3.10,26.35,17.45,27.76,9.95,6.30,72.84,9.88,100.77,NaN,154.0,Moderate,2,False,False,90
3,Bengaluru,2015-03-24,60.47,77.36,5.39,29.87,20.88,35.10,1.46,6.07,64.12,5.90,61.48,NaN,119.0,Moderate,2,False,False,90
4,Bengaluru,2015-03-25,62.56,77.36,3.16,23.57,16.39,27.13,10.05,4.98,82.34,4.53,39.99,NaN,232.0,Poor,2,True,False,75


In [2]:
feature_df["year"] = feature_df["Date"].dt.year
feature_df["month"] = feature_df["Date"].dt.month
feature_df["day_of_week"] = feature_df["Date"].dt.dayofweek
feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype(int)

display(
    feature_df[
        [
            "City", "Date", "year", "month",
            "day_of_week", "is_weekend"
        ]
    ].head(10)
)

,City,Date,year,month,day_of_week,is_weekend
0,Bengaluru,2015-03-21,2015,3,5,1
1,Bengaluru,2015-03-22,2015,3,6,1
2,Bengaluru,2015-03-23,2015,3,0,0
3,Bengaluru,2015-03-24,2015,3,1,0
4,Bengaluru,2015-03-25,2015,3,2,0
5,Bengaluru,2015-03-26,2015,3,3,0
6,Bengaluru,2015-03-27,2015,3,4,0
7,Bengaluru,2015-03-28,2015,3,5,1
8,Bengaluru,2015-03-29,2015,3,6,1
9,Bengaluru,2015-03-30,2015,3,0,0


In [3]:
city_groups = feature_df.groupby("City")

feature_df["aqi_lag_1"] = city_groups["AQI"].shift(1)
feature_df["aqi_lag_7"] = city_groups["AQI"].shift(7)

feature_df["aqi_rolling_mean_3"] = city_groups["AQI"].transform(
    lambda values: values.rolling(window=3, min_periods=3).mean()
)

feature_df["aqi_rolling_mean_7"] = city_groups["AQI"].transform(
    lambda values: values.rolling(window=7, min_periods=7).mean()
)

feature_df["pm25_lag_1"] = city_groups["PM2.5"].shift(1)
feature_df["pm10_lag_1"] = city_groups["PM10"].shift(1)

display(
    feature_df[
        [
            "City", "Date", "AQI",
            "aqi_lag_1", "aqi_lag_7",
            "aqi_rolling_mean_3", "aqi_rolling_mean_7",
            "pm25_lag_1", "pm10_lag_1"
        ]
    ].head(10)
)

,City,Date,AQI,aqi_lag_1,aqi_lag_7,aqi_rolling_mean_3,aqi_rolling_mean_7,pm25_lag_1,pm10_lag_1
0,Bengaluru,2015-03-21,91.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Bengaluru,2015-03-22,120.0,91.0,NaN,NaN,NaN,48.59,77.36
2,Bengaluru,2015-03-23,154.0,120.0,NaN,121.666667,NaN,47.38,77.36
3,Bengaluru,2015-03-24,119.0,154.0,NaN,131.000000,NaN,65.65,77.36
4,Bengaluru,2015-03-25,232.0,119.0,NaN,168.333333,NaN,60.47,77.36
5,Bengaluru,2015-03-26,132.0,232.0,NaN,161.000000,NaN,62.56,77.36
6,Bengaluru,2015-03-27,123.0,132.0,NaN,162.333333,138.714286,60.73,77.36
7,Bengaluru,2015-03-28,152.0,123.0,91.0,135.666667,147.428571,42.10,77.36
8,Bengaluru,2015-03-29,143.0,152.0,120.0,139.333333,150.714286,40.00,77.36
9,Bengaluru,2015-03-30,80.0,143.0,154.0,125.000000,140.142857,32.16,77.36


In [4]:
city_groups = feature_df.groupby("City")

next_observation_date = city_groups["Date"].shift(-1)

feature_df["next_day_aqi"] = city_groups["AQI"].shift(-1)

is_actual_next_day = (
    next_observation_date
    == feature_df["Date"] + pd.Timedelta(days=1)
)

feature_df.loc[
    ~is_actual_next_day,
    "next_day_aqi"
] = np.nan

print(
    "Rows with an actual next-day AQI target:",
    feature_df["next_day_aqi"].notna().sum()
)

display(
    feature_df[
        [
            "City", "Date", "AQI",
            "aqi_lag_1",
            "aqi_rolling_mean_3",
            "next_day_aqi"
        ]
    ].head(10)
)

Rows with an actual next-day AQI target: 4663


,City,Date,AQI,aqi_lag_1,aqi_rolling_mean_3,next_day_aqi
0,Bengaluru,2015-03-21,91.0,NaN,NaN,120.0
1,Bengaluru,2015-03-22,120.0,91.0,NaN,154.0
2,Bengaluru,2015-03-23,154.0,120.0,121.666667,119.0
3,Bengaluru,2015-03-24,119.0,154.0,131.000000,232.0
4,Bengaluru,2015-03-25,232.0,119.0,168.333333,132.0
5,Bengaluru,2015-03-26,132.0,232.0,161.000000,123.0
6,Bengaluru,2015-03-27,123.0,132.0,162.333333,152.0
7,Bengaluru,2015-03-28,152.0,123.0,135.666667,143.0
8,Bengaluru,2015-03-29,143.0,152.0,139.333333,80.0
9,Bengaluru,2015-03-30,80.0,143.0,125.000000,90.0


In [5]:
required_model_columns = [
    "AQI",
    "aqi_lag_1",
    "aqi_lag_7",
    "aqi_rolling_mean_3",
    "aqi_rolling_mean_7",
    "pm25_lag_1",
    "pm10_lag_1",
    "next_day_aqi"
]

rows_before_model_filter = len(feature_df)

model_ready_df = (
    feature_df
    .dropna(subset=required_model_columns)
    .copy()
)

print("Rows before model-ready filtering:", rows_before_model_filter)
print("Rows after model-ready filtering:", len(model_ready_df))
print("Rows removed due to missing target/history:", 
      rows_before_model_filter - len(model_ready_df))

print("\nModel-ready rows by city:")
display(model_ready_df["City"].value_counts().to_frame(name="Records"))

display(
    model_ready_df[
        [
            "City", "Date", "AQI",
            "aqi_lag_1", "aqi_lag_7",
            "aqi_rolling_mean_3", "aqi_rolling_mean_7",
            "pm25_lag_1", "pm10_lag_1",
            "next_day_aqi"
        ]
    ].head()
)

Rows before model-ready filtering: 4684
Rows after model-ready filtering: 4642
Rows removed due to missing target/history: 42

Model-ready rows by city:


,Records
City,
Delhi,1986
Bengaluru,1894
Mumbai,762


,City,Date,AQI,aqi_lag_1,aqi_lag_7,aqi_rolling_mean_3,aqi_rolling_mean_7,pm25_lag_1,pm10_lag_1,next_day_aqi
7,Bengaluru,2015-03-28,152.0,123.0,91.0,135.666667,147.428571,42.10,77.36,143.0
8,Bengaluru,2015-03-29,143.0,152.0,120.0,139.333333,150.714286,40.00,77.36,80.0
9,Bengaluru,2015-03-30,80.0,143.0,154.0,125.000000,140.142857,32.16,77.36,90.0
10,Bengaluru,2015-03-31,90.0,80.0,119.0,104.333333,136.000000,30.78,77.36,151.0
11,Bengaluru,2015-04-01,151.0,90.0,232.0,107.000000,124.428571,38.99,77.36,123.0


# Feature Engineering Conclusion

- Calendar features were created from the observation date: year, month, day_of_week, and is_weekend.
- AQI lag features and PM2.5/PM10 lag features were created separately for each city.
- Rolling AQI means captured recent AQI trends using 3-record and 7-record windows.
- The prediction target, next_day_aqi, was created only when the next available observation was exactly one calendar day later.
- Rows without sufficient historical values or a true next-day target were removed from the model-ready dataset.
- The final dataset contains 4,642 model-ready observations.
- The feature-engineering process avoids target leakage because no feature uses information from after the prediction date.